# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shakhaoathpappu-jpg/FlyRank-ML-Internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: Which published pages are likely to need a content refresh — based on staleness (time since last update) and how far their CTR falls short of what similarly-ranked pages typically get?

Decision this supports: Given a limited content-ops budget, which pages should be prioritized for refresh this month, and why (so the reasoning is auditable, not a black box).

# 2. Data

Which release, which tables, date windows, what you excluded and why. Public-safe.

Release: FlyRank ML Internship warehouse (Hugging Face, FlyRank/internship-warehouse).
Tables used: fact_content_daily_performance (daily grain, aggregated to page-month) and dim_content (content metadata: created/updated dates, publish status).
Date window: 2025-01 through 2026-06 (18 months). 2025-01–2026-05 used for training/validation; 2026-06 is the dataset's sealed final month, held out and used only once for final honest evaluation.
Excluded: client names, URLs, and raw search queries (all identifiers stay hashed); unpublished or deleted content; any column derived from a future month relative to a given decision point.

In [1]:
import gc
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
REPO_ID = "FlyRank/internship-warehouse"

ALL_TRAIN_VAL_MONTHS = [f"2025-{m:02d}" for m in range(1, 13)] + [f"2026-{m:02d}" for m in range(1, 6)]
SEALED_TEST_MONTH = "2026-06"
ALL_MONTHS = ALL_TRAIN_VAL_MONTHS + [SEALED_TEST_MONTH]

dim_content_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename="dim_content.parquet", token=HF_TOKEN)
dim_content = pd.read_parquet(dim_content_path)
dim_content["content_created_date"] = pd.to_datetime(dim_content["content_created_date"], errors="coerce")
dim_content["content_updated_date"] = pd.to_datetime(dim_content["content_updated_date"], errors="coerce")

def aggregate_month(df, month):
    m = (
        df.groupby(["client_hash_id", "content_hash_id"], as_index=False)
        .agg(
            gsc_clicks=("gsc_clicks", "sum"),
            gsc_impressions=("gsc_impressions", "sum"),
            gsc_sum_position=("gsc_sum_position", "sum"),
            days_with_gsc=("gsc_data_available", "sum"),
        )
    )
    m["month"] = month
    impressions_f = m["gsc_impressions"].astype(float).replace(0, np.nan)
    m["ctr"] = m["gsc_clicks"].astype(float) / impressions_f
    m["avg_position"] = m["gsc_sum_position"].astype(float) / impressions_f
    return m[m["days_with_gsc"] > 0].copy()

def load_and_aggregate(month):
    path = hf_hub_download(
        repo_id=REPO_ID, repo_type="dataset",
        filename=f"fact_content_daily_performance/month={month}/data_0.parquet",
        token=HF_TOKEN
    )
    raw = pd.read_parquet(path)
    agg = aggregate_month(raw, month)
    del raw
    gc.collect()
    return agg

panel_parts = []
for m in ALL_MONTHS:
    print("Loading + aggregating", m, "...")
    part = load_and_aggregate(m)
    panel_parts.append(part)
    print(f"  -> {len(part)} page-month rows")

panel_df = pd.concat(panel_parts, ignore_index=True)
del panel_parts
gc.collect()

print("\n=== FULL PANEL ===")
print("Shape:", panel_df.shape)
print(panel_df["month"].value_counts().sort_index())
print("\nUnique pages overall:", panel_df["content_hash_id"].nunique())
print("Unique clients overall:", panel_df["client_hash_id"].nunique())

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Loading + aggregating 2025-01 ...


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

  -> 237910 page-month rows
Loading + aggregating 2026-06 ...


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

  -> 208636 page-month rows

=== FULL PANEL ===
Shape: (1546456, 9)
month
2025-01       476
2025-02      5903
2025-03     10374
2025-04     13046
2025-05     14887
2025-06     16399
2025-07     27945
2025-08     37204
2025-09     53127
2025-10     64906
2025-11     99403
2025-12    109639
2026-01    121544
2026-02    153559
2026-03    176738
2026-04    194760
2026-05    237910
2026-06    208636
Name: count, dtype: int64

Unique pages overall: 309234
Unique clients overall: 67


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Assumptions: a page "needs refresh" if its clicks in the month after the decision point are lower than in the decision month itself. All features are computed strictly from the decision month or earlier.
Features: gsc_clicks, gsc_impressions, ctr, avg_position, staleness_score (days since last content update, capped at 180 days), ctr_gap (how far actual CTR falls below the median CTR for that position bucket).
Baseline: the ML-07 rule-based score — 0.5*staleness_score + 0.5*ctr_gap — with no training required.
Validation design: time-aware split, no random shuffling — train on the earliest decision months, validate on the middle months, and evaluate once, honestly, on the sealed final decision month (2026-05 → outcome 2026-06).
Leakage checks: decision-month ranges for train/val/test are strictly non-overlapping in time; the label itself (future_clicks) is never used as a feature.

In [1]:
MONTH_ORDER = ALL_MONTHS  # already chronological: 2025-01 ... 2026-06
month_to_idx = {m: i for i, m in enumerate(MONTH_ORDER)}

def next_month(m):
    idx = month_to_idx[m]
    return MONTH_ORDER[idx + 1] if idx + 1 < len(MONTH_ORDER) else None

def position_bucket(p):
    if pd.isna(p): return "unknown"
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

def decision_date(month_str):
    return pd.Period(month_str, freq="M").end_time.normalize()

def build_pairs(decision_months):
    frames = []
    for dm in decision_months:
        om = next_month(dm)
        cur = panel_df[panel_df["month"] == dm].copy()
        nxt = panel_df[panel_df["month"] == om][["client_hash_id", "content_hash_id", "gsc_clicks"]] \
              .rename(columns={"gsc_clicks": "future_clicks"})
        merged = cur.merge(nxt, on=["client_hash_id", "content_hash_id"], how="inner")
        merged["decision_month"] = dm
        merged["outcome_month"] = om
        frames.append(merged)
    return pd.concat(frames, ignore_index=True)

def add_features(df):
    df = df.merge(
        dim_content[["client_hash_id", "content_hash_id", "content_created_date", "content_updated_date",
                      "is_published", "is_deleted", "content_type"]],
        on=["client_hash_id", "content_hash_id"], how="left"
    )
    df["decision_date"] = df["decision_month"].apply(decision_date)
    df = df[(df["is_published"] == True) & (df["is_deleted"] == False)]
    df = df[df["content_created_date"].isna() | (df["content_created_date"] <= df["decision_date"])]

    df["days_since_update"] = (df["decision_date"] - df["content_updated_date"]).dt.days
    df.loc[df["days_since_update"] < 0, "days_since_update"] = np.nan

    df["position_bucket"] = df["avg_position"].apply(position_bucket)
    bucket_median_ctr = df.groupby("position_bucket")["ctr"].transform("median")
    df["ctr_gap"] = (1 - (df["ctr"] / bucket_median_ctr)).clip(lower=0, upper=1).fillna(0)

    STALE_CAP_DAYS = 180
    df["staleness_score"] = (df["days_since_update"].clip(upper=STALE_CAP_DAYS) / STALE_CAP_DAYS).fillna(0)
    df["needs_refresh"] = (df["future_clicks"] < df["gsc_clicks"]).astype(int)
    return df

# Non-overlapping, chronologically ordered splits
TRAIN_DECISION_MONTHS = MONTH_ORDER[0:9]    # 2025-01 .. 2025-09
VAL_DECISION_MONTHS   = MONTH_ORDER[9:16]   # 2025-10 .. 2026-04
TEST_DECISION_MONTH   = MONTH_ORDER[16]     # 2026-05  (outcome = sealed 2026-06)

train_df = add_features(build_pairs(TRAIN_DECISION_MONTHS))
val_df   = add_features(build_pairs(VAL_DECISION_MONTHS))
test_df  = add_features(build_pairs([TEST_DECISION_MONTH]))

print("train:", train_df.shape, "positive rate:", round(train_df["needs_refresh"].mean(), 4))
print("val:  ", val_df.shape,   "positive rate:", round(val_df["needs_refresh"].mean(), 4))
print("test: ", test_df.shape,  "positive rate:", round(test_df["needs_refresh"].mean(), 4))

# Leakage / time-ordering check
print("\nTrain decision months:", sorted(train_df['decision_month'].unique()))
print("Val decision months:  ", sorted(val_df['decision_month'].unique()))
print("Test decision month:  ", sorted(test_df['decision_month'].unique()))
assert max(train_df['decision_month']) < min(val_df['decision_month']), "Train/val time overlap!"
assert max(val_df['decision_month']) < min(test_df['decision_month']), "Val/test time overlap!"
print("\nNo time overlap between splits — confirmed.")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

The honest table below compares the rule-based baseline to a trained model on the same, never-touched-until-now sealed test split.

In [1]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score

FEATURE_COLS = ["gsc_clicks", "gsc_impressions", "ctr", "avg_position", "staleness_score", "ctr_gap"]

def prep_X(df):
    return df[FEATURE_COLS].fillna(0)

X_train, y_train = prep_X(train_df), train_df["needs_refresh"]
X_val, y_val = prep_X(val_df), val_df["needs_refresh"]
X_test, y_test = prep_X(test_df), test_df["needs_refresh"]

baseline_val_score = 0.5 * val_df["staleness_score"] + 0.5 * val_df["ctr_gap"]
baseline_test_score = 0.5 * test_df["staleness_score"] + 0.5 * test_df["ctr_gap"]
baseline_val_auc = roc_auc_score(y_val, baseline_val_score)
baseline_test_auc = roc_auc_score(y_test, baseline_test_score)

model = GradientBoostingClassifier(random_state=42, n_estimators=200, max_depth=3, learning_rate=0.05)
model.fit(X_train, y_train)

model_val_score = model.predict_proba(X_val)[:, 1]
model_test_score = model.predict_proba(X_test)[:, 1]
model_val_auc = roc_auc_score(y_val, model_val_score)
model_test_auc = roc_auc_score(y_test, model_test_score)

results_table = pd.DataFrame({
    "split": ["validation", "validation", "sealed_test", "sealed_test"],
    "approach": ["baseline (rule)", "model (GBM)", "baseline (rule)", "model (GBM)"],
    "auc": [baseline_val_auc, model_val_auc, baseline_test_auc, model_test_auc],
    "n": [len(y_val), len(y_val), len(y_test), len(y_test)],
    "positive_rate": [y_val.mean(), y_val.mean(), y_test.mean(), y_test.mean()],
})
print(results_table)

feature_importance = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nFeature importances:\n", feature_importance)

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [1]:
test_scored = test_df.copy()
test_scored["model_score"] = model_test_score
test_scored["baseline_score"] = baseline_test_score

def reason_code(row):
    if row["staleness_score"] >= 0.5 and row["ctr_gap"] >= 0.5:
        return "STALE_AND_CTR_GAP"
    if row["staleness_score"] > row["ctr_gap"]:
        return "STALE"
    if row["ctr_gap"] > row["staleness_score"]:
        return "CTR_GAP"
    return "NO_SIGNAL"

test_scored["reason_code"] = test_scored.apply(reason_code, axis=1)

def action_label(score):
    if score >= 0.66: return "refresh_now"
    if score >= 0.33: return "monitor"
    return "no_action"

test_scored["action_label"] = test_scored["model_score"].apply(action_label)

recommendations = test_scored.sort_values("model_score", ascending=False).reset_index(drop=True)
recommendations["rank"] = recommendations.index + 1

rec_cols = ["rank", "client_hash_id", "content_hash_id", "model_score", "baseline_score", "reason_code",
            "action_label", "gsc_clicks", "gsc_impressions", "ctr", "avg_position", "days_since_update"]

import os
os.makedirs("work/outputs", exist_ok=True)
recommendations[rec_cols].to_csv("work/outputs/capstone_ranked_recommendations.csv", index=False)

print(recommendations["action_label"].value_counts())
recommendations[rec_cols].head(20)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [1]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

os.makedirs("work/outputs/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(6, 4))
results_table.pivot(index="split", columns="approach", values="auc").plot(kind="bar", ax=ax)
ax.set_ylabel("AUC"); ax.set_title("Baseline vs Model AUC")
plt.tight_layout(); plt.savefig("work/outputs/figures/auc_comparison.png", dpi=150); plt.show()

fpr_b, tpr_b, _ = roc_curve(y_test, baseline_test_score)
fpr_m, tpr_m, _ = roc_curve(y_test, model_test_score)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr_b, tpr_b, label=f"Baseline (AUC={baseline_test_auc:.3f})")
ax.plot(fpr_m, tpr_m, label=f"Model (AUC={model_test_auc:.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC — sealed test (2026-05 → 2026-06)"); ax.legend()
plt.tight_layout(); plt.savefig("work/outputs/figures/roc_sealed_test.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
feature_importance.plot(kind="barh", ax=ax)
ax.set_title("Feature importance (Gradient Boosting)")
plt.tight_layout(); plt.savefig("work/outputs/figures/feature_importance.png", dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
recommendations["action_label"].value_counts().plot(kind="bar", ax=ax)
ax.set_title("Recommended actions (sealed test month)")
plt.tight_layout(); plt.savefig("work/outputs/figures/action_distribution.png", dpi=150); plt.show()

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
